In [1]:
from amplpy import AMPL
import itertools

# Generate vectors as per your previous function
def generate_valid_vectors(N):
    """Generate all 0/1 vectors with the constraint: if vec[i] == 1, then vec[N+i] must be 1"""
    for vec in itertools.product([0, 1], repeat=2*N):
        if all(not (vec[i] == 1 and vec[N+i] == 0) for i in range(N)):
            yield vec

# Load your AMPL model
ampl = AMPL()
ampl.read("/home/kkishan/Downloads/alan.mod")  # Adjust path

# Extract integer/binary variable names for indexing
int_var_names = []
for var in ampl.getVariables():
    var_obj = var[1] if isinstance(var, tuple) else var
    int_var_names.append(var_obj.name())

N = len(int_var_names)
print(f"Total integer/binary variables: {N}")

# Set solver to Gurobi
ampl.setOption('solver', 'gurobi')

# Iterate over generated vectors emulating branching nodes
for idx, vec in enumerate(generate_valid_vectors(N)):
    print(f"\n=== Node {idx+1} with bounds vector: {vec} ===")

    fixed_vars = []
    # Fix variable bounds according to vector
    for i, var_name in enumerate(int_var_names):
        var = ampl.getVariable(var_name)
        lb, ub = var.lb(), var.ub()

        if vec[i] == 1:         # Fix to lower bound
            var.fix(lb)
            fixed_vars.append(var)
        elif vec[N + i] == 1:   # Fix to upper bound
            var.fix(ub)
            fixed_vars.append(var)
        else:
            var.unfix()  # Make sure unfixed if neither bound fixed

    # Solve this node relaxation (no NodeLimit, full solve but fixing bounds mimics node)
    ampl.setOption('gurobi_options', 'NodeLimit=0')  # Only root relaxation for node
    ampl.solve()

    # Display variable bounds and relaxation solution values
    for var in ampl.getVariables():
        var_obj = var[1] if isinstance(var, tuple) else var
        print(f"{var_obj.name()} -> LB: {var_obj.lb()}, UB: {var_obj.ub()}, Relaxed value: {var_obj.value()}")

    # Unfix all variables for next node
    for var in fixed_vars:
        var.unfix()


Total integer/binary variables: 5

=== Node 1 with bounds vector: (0, 0, 0, 0, 0, 0, 0, 0, 0, 0) ===
Gurobi 12.0.3:   lim:nodes = 0
Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
x1 -> LB: 0.0, UB: 1.0, Relaxed value: 0.0
x2 -> LB: 0.0, UB: 1.0, Relaxed value: 1.0
x3 -> LB: 0.0, UB: 1.0, Relaxed value: 1.0
x4 -> LB: 0.0, UB: 1.0, Relaxed value: 0.0
x5 -> LB: 0.0, UB: 1.0, Relaxed value: 1.0

=== Node 2 with bounds vector: (0, 0, 0, 0, 0, 0, 0, 0, 0, 1) ===
Gurobi 12.0.3:   lim:nodes = 0
Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
x1 -> LB: 0.0, UB: 1.0, Relaxed value: 0.0
x2 -> LB: 0.0, UB: 1.0, Relaxed value: 1.0
x3 -> LB: 0.0, UB: 1.0, Relaxed value: 1.0
x4 -> LB: 0.0, UB: 1.0, Relaxed value: 0.0
x5 -> LB: 1.0, UB: 1.0, Relaxed value: 1.0

=== Node 3 with bounds vector: (0, 0, 0, 0, 0, 0, 0, 0, 1, 0) ===
Gurobi 12.0.3:   lim:nodes = 0
Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
x1 -> LB: 0.0, UB: 1.0, Relaxed value: 1.